# ⚙️ 04 – Pivot, Merge et Export avec Pandas

---

## 🎯 Objectifs
- Créer un **tableau croisé dynamique** avec `pivot_table()` — équivalent Excel
- **Fusionner** plusieurs tableaux avec `merge()` — les 4 types de jointures
- **Exporter** les résultats en CSV, Excel et JSON

> ℹ️ **Astuce** : Les corrections sont cachées dans des blocs pliables.  
> Cliquez sur **💡 Correction** pour dérouler la solution.

In [ ]:
import pandas as pd
print("Pandas version :", pd.__version__)

---
## 📝 Partie 1 – `pivot_table()` : le tableau croisé dynamique

### 🔎 Qu'est-ce qu'un tableau croisé dynamique ?

Un **tableau croisé dynamique** (TCD) permet de **réorganiser et résumer** des données en croisant deux dimensions. Dans Excel, c'est l'outil le plus utilisé par les analystes. Pandas le fait avec `pivot_table()`.

**Exemple concret :** vous avez des ventes de pizzas par ville et par saveur, et vous voulez voir en un coup d'œil combien de chaque saveur a été vendue dans chaque ville.

```
Données brutes (format "long") :      Après pivot_table (format "large") :

Ville      Saveur   Qté               Saveur   →  Margherita  Reine  Végé
Paris      Reine     10               Ville ↓
Lyon       Margherita 8               Lyon             8       12      0
Paris      Margherita 5               Marseille        0        6      9
Lyon       Reine     12               Paris            5       10      7
Paris      Végé       7
Marseille  Reine      6
Marseille  Végé       9
```

### 🔎 Les paramètres de `pivot_table()`

```python
df.pivot_table(
    values   = "Qté",          # la colonne à calculer (les chiffres)
    index    = "Ville",         # ce qui devient les LIGNES
    columns  = "Saveur",        # ce qui devient les COLONNES
    aggfunc  = "sum",           # comment on agrège : sum, mean, count, max...
    fill_value = 0              # remplacer les cases vides par 0
)
```

**Équivalent Excel :** Insertion → Tableau croisé dynamique → glisser les champs dans les zones Lignes, Colonnes, Valeurs.

| Paramètre | Zone Excel TCD | Description |
|-----------|---------------|-------------|
| `values` | Zone **Valeurs** | La donnée à calculer |
| `index` | Zone **Lignes** | Ce qui s'affiche en ligne |
| `columns` | Zone **Colonnes** | Ce qui s'affiche en colonne |
| `aggfunc` | Type de calcul | sum, mean, count, max, min… |
| `fill_value` | — | Remplace les NaN (cases vides) |

In [ ]:
import pandas as pd

# Données de ventes : chaque ligne = 1 transaction
ventes = pd.DataFrame({
    "Ville"  : ["Paris", "Lyon", "Paris", "Lyon", "Paris", "Marseille", "Marseille"],
    "Saveur" : ["Reine", "Margherita", "Margherita", "Reine", "Végé", "Reine", "Végé"],
    "Qté"    : [10, 8, 5, 12, 7, 6, 9],
    "CA"     : [120, 80, 50, 120, 63, 60, 81]
})

print("=== Données brutes ===")
print(ventes)

print()

# pivot_table : Villes en lignes, Saveurs en colonnes, somme des quantités
pivot_qte = ventes.pivot_table(
    values   = "Qté",
    index    = "Ville",
    columns  = "Saveur",
    aggfunc  = "sum",
    fill_value = 0        # les cases vides (pas de cette saveur dans cette ville) → 0
)

print("=== Tableau croisé : Qté par Ville × Saveur ===")
print(pivot_qte)

In [ ]:
import pandas as pd

ventes = pd.DataFrame({
    "Ville"  : ["Paris", "Lyon", "Paris", "Lyon", "Paris", "Marseille", "Marseille"],
    "Saveur" : ["Reine", "Margherita", "Margherita", "Reine", "Végé", "Reine", "Végé"],
    "Qté"    : [10, 8, 5, 12, 7, 6, 9],
    "CA"     : [120, 80, 50, 120, 63, 60, 81]
})

# Pivot sur le CA moyen (aggfunc="mean")
pivot_ca = ventes.pivot_table(
    values   = "CA",
    index    = "Ville",
    columns  = "Saveur",
    aggfunc  = "mean",
    fill_value = 0
)

print("=== Tableau croisé : CA moyen par Ville × Saveur ===")
print(pivot_ca)

print()

# Totaux par ligne et colonne
pivot_total = ventes.pivot_table(
    values   = "Qté",
    index    = "Ville",
    columns  = "Saveur",
    aggfunc  = "sum",
    fill_value = 0,
    margins  = True,      # ajoute une ligne et colonne "All" avec les totaux
    margins_name = "Total"
)

print("=== Avec les totaux (margins=True) ===")
print(pivot_total)

---
## 📝 Partie 2 – `merge()` : fusionner des tableaux

### 🔎 Pourquoi fusionner des tableaux ?

Dans la vraie vie, les données sont rarement dans un seul fichier. On a souvent :
- Un fichier **clients** avec les infos personnelles
- Un fichier **commandes** avec les achats
- Un fichier **produits** avec les prix

Pour analyser "quel client a dépensé combien sur quel produit", il faut **fusionner** ces fichiers. C'est ce que fait `merge()` — l'équivalent d'un `JOIN` en SQL ou d'un `RECHERCHEV()` dans Excel.

### 🔎 Les 4 types de jointures

Le paramètre `how` contrôle ce qu'on garde quand des lignes n'ont pas de correspondance :

```
Tableau A (Employés) :    Tableau B (Salaires) :    Correspondances :
id  Nom                   id  Salaire               A.id=1 ↔ B.id=1  ✓
 1  Alice                  1   3200                 A.id=2 ↔ B.id=2  ✓
 2  Bob                    2   4100                 A.id=3 → pas dans B
 3  Chloé                  4   5000                 B.id=4 → pas dans A

how="inner"  → garde seulement les correspondances : Alice, Bob          (2 lignes)
how="left"   → garde tout A + correspondances B   : Alice, Bob, Chloé   (3 lignes, Chloé sans salaire)
how="right"  → garde tout B + correspondances A   : Alice, Bob, +id 4   (3 lignes, id 4 sans nom)
how="outer"  → garde TOUT                         : Alice, Bob, Chloé, +id 4 (4 lignes)
```

| `how` | Équivalent SQL | Ce qu'on garde |
|-------|---------------|----------------|
| `"inner"` | `INNER JOIN` | Seulement les lignes **présentes dans les deux** tableaux |
| `"left"` | `LEFT JOIN` | Tout le tableau de **gauche**, + ce qui correspond à droite |
| `"right"` | `RIGHT JOIN` | Tout le tableau de **droite**, + ce qui correspond à gauche |
| `"outer"` | `FULL OUTER JOIN` | **Tout**, des deux côtés — les manquants deviennent NaN |

> 💡 **Le plus courant en pratique :** `inner` (on ne garde que ce qui correspond) et `left` (on garde tout le tableau principal + on enrichit avec le second).

In [ ]:
import pandas as pd

# Tableau A : employés
employes = pd.DataFrame({
    "id" : [1, 2, 3],
    "Nom": ["Alice", "Bob", "Chloé"]
})

# Tableau B : salaires (id 3 absent, id 4 en plus)
salaires = pd.DataFrame({
    "id"     : [1, 2, 4],
    "Salaire": [3200, 4100, 5000]
})

print("Tableau A — Employés :")
print(employes)
print("\nTableau B — Salaires :")
print(salaires)

print()

# inner : seulement les id présents dans les DEUX tableaux (1 et 2)
inner = pd.merge(employes, salaires, on="id", how="inner")
print("=== inner : correspondances uniquement ===")
print(inner)
print(f"→ {len(inner)} lignes (Chloé sans salaire et id 4 sans nom sont exclus)")

In [ ]:
import pandas as pd

employes = pd.DataFrame({"id": [1, 2, 3], "Nom": ["Alice", "Bob", "Chloé"]})
salaires = pd.DataFrame({"id": [1, 2, 4], "Salaire": [3200, 4100, 5000]})

# left : tout employes + ce qui correspond dans salaires
left = pd.merge(employes, salaires, on="id", how="left")
print("=== left : tout le tableau gauche ===")
print(left)
print("→ Chloé (id=3) est gardée mais son Salaire = NaN")

print()

# right : tout salaires + ce qui correspond dans employes
right = pd.merge(employes, salaires, on="id", how="right")
print("=== right : tout le tableau droit ===")
print(right)
print("→ id=4 est gardé mais son Nom = NaN")

print()

# outer : TOUT des deux tableaux
outer = pd.merge(employes, salaires, on="id", how="outer")
print("=== outer : tout des deux côtés ===")
print(outer)
print("→ 4 lignes : les manquants remplis par NaN")

---
## 📝 Partie 3 – Exporter les résultats

### 🔎 Sauvegarder un DataFrame

Une fois l'analyse terminée, on exporte souvent les résultats pour les partager ou les intégrer dans d'autres outils.

| Format | Méthode | Quand l'utiliser |
|--------|---------|------------------|
| **CSV** | `df.to_csv("fichier.csv", index=False)` | Format universel — compatible avec tout |
| **Excel** | `df.to_excel("fichier.xlsx", index=False)` | Pour envoyer à des collègues qui utilisent Excel |
| **JSON** | `df.to_json("fichier.json", orient="records")` | Pour des API web ou des applications |

**Le paramètre `index=False` :** par défaut, Pandas ajoute une colonne d'index (0, 1, 2…) au moment de l'export. `index=False` supprime cette colonne parasite dans le fichier exporté.

**Le paramètre `orient="records"` pour JSON :** définit la structure du JSON. `"records"` crée une liste d'objets JSON, un par ligne — c'est le format le plus lisible et le plus compatible avec les APIs.

```json
// orient="records"
[
  {"id": 1, "Nom": "Alice", "Salaire": 3200},
  {"id": 2, "Nom": "Bob",   "Salaire": 4100}
]
```

In [ ]:
import pandas as pd

# Créer un tableau de résultats à exporter
employes = pd.DataFrame({"id": [1, 2, 3], "Nom": ["Alice", "Bob", "Chloé"]})
salaires = pd.DataFrame({"id": [1, 2, 3], "Salaire": [3200, 4100, 2900], "Service": ["IT", "RH", "IT"]})

resultat = pd.merge(employes, salaires, on="id", how="inner")
print("=== Tableau à exporter ===")
print(resultat)

print()

# Export CSV — le plus courant
resultat.to_csv("resultat.csv", index=False)
print("✅ CSV exporté : resultat.csv")

# Export Excel
resultat.to_excel("resultat.xlsx", index=False)
print("✅ Excel exporté : resultat.xlsx")

# Export JSON
resultat.to_json("resultat.json", orient="records", indent=2)
print("✅ JSON exporté : resultat.json")

print()
print("Vérification — relire le CSV exporté :")
print(pd.read_csv("resultat.csv"))

---
## 🧩 Exercice 1 – Tableau croisé dynamique

Vous avez des données de ventes d'une chaîne de restaurants :

```python
ventes = pd.DataFrame({
    "Ville"   : ["Paris", "Lyon", "Paris", "Lyon", "Paris",
                 "Marseille", "Marseille", "Lyon", "Paris"],
    "Produit" : ["Pizza", "Pizza", "Burger", "Burger", "Sushi",
                 "Pizza", "Burger", "Sushi", "Pizza"],
    "Qté"     : [15, 10, 8, 12, 6, 20, 9, 5, 18],
    "CA"      : [187, 125, 72, 108, 90, 250, 81, 75, 225]
})
```

1. Créez un pivot_table avec **Ville en lignes**, **Produit en colonnes**, **somme de Qté** en valeurs
2. Ajoutez `fill_value=0` et `margins=True` pour voir les totaux
3. Créez un second pivot avec le **CA moyen** (aggfunc="mean") par Ville × Produit
4. Dans quelle ville et pour quel produit le CA est-il le plus élevé ?
5. Exportez le premier pivot en CSV sous le nom `"ventes_pivot.csv"`

In [ ]:
import pandas as pd

ventes = pd.DataFrame({
    "Ville"   : ["Paris", "Lyon", "Paris", "Lyon", "Paris",
                 "Marseille", "Marseille", "Lyon", "Paris"],
    "Produit" : ["Pizza", "Pizza", "Burger", "Burger", "Sushi",
                 "Pizza", "Burger", "Sushi", "Pizza"],
    "Qté"     : [15, 10, 8, 12, 6, 20, 9, 5, 18],
    "CA"      : [187, 125, 72, 108, 90, 250, 81, 75, 225]
})

# TODO 1 : pivot Ville × Produit, somme Qté
pivot_qte = ventes.pivot_table(
    values   = ...,
    index    = ...,
    columns  = ...,
    aggfunc  = ...,
    fill_value = ...
)
print("1. Pivot Qté :")
print(pivot_qte)

# TODO 2 : ajouter les totaux
pivot_totaux = ventes.pivot_table(
    values="Qté", index="Ville", columns="Produit",
    aggfunc="sum", fill_value=0,
    margins=..., margins_name=...
)
print("\n2. Avec totaux :")
print(pivot_totaux)

# TODO 3 : pivot CA moyen
pivot_ca = ventes.pivot_table(
    values=..., index=..., columns=...,
    aggfunc=..., fill_value=0
)
print("\n3. Pivot CA moyen :")
print(pivot_ca.round(1))

# TODO 4 : ville et produit avec le CA le plus élevé
ca_total = ventes.groupby(["Ville", "Produit"])["CA"].sum()
print(f"\n4. Meilleur CA : {ca_total.idxmax()} → {ca_total.max()} €")

# TODO 5 : export CSV
pivot_qte.to_csv(..., index=...)
print("\n5. ✅ Pivot exporté en CSV")

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

ventes = pd.DataFrame({
    "Ville"   : ["Paris", "Lyon", "Paris", "Lyon", "Paris",
                 "Marseille", "Marseille", "Lyon", "Paris"],
    "Produit" : ["Pizza", "Pizza", "Burger", "Burger", "Sushi",
                 "Pizza", "Burger", "Sushi", "Pizza"],
    "Qté"     : [15, 10, 8, 12, 6, 20, 9, 5, 18],
    "CA"      : [187, 125, 72, 108, 90, 250, 81, 75, 225]
})

# 1
pivot_qte = ventes.pivot_table(
    values="Qté", index="Ville", columns="Produit",
    aggfunc="sum", fill_value=0
)
print(pivot_qte)

# 2 — margins=True ajoute la ligne/colonne Total
pivot_totaux = ventes.pivot_table(
    values="Qté", index="Ville", columns="Produit",
    aggfunc="sum", fill_value=0,
    margins=True, margins_name="Total"
)
print(pivot_totaux)
# Paris est la ville avec le plus de ventes (47)

# 3
pivot_ca = ventes.pivot_table(
    values="CA", index="Ville", columns="Produit",
    aggfunc="mean", fill_value=0
)
print(pivot_ca.round(1))

# 4 — groupby multi-colonnes pour trouver le meilleur CA
ca_total = ventes.groupby(["Ville", "Produit"])["CA"].sum()
print(f"Meilleur CA : {ca_total.idxmax()} → {ca_total.max()} €")
# → (Paris, Pizza) : 187 + 225 = 412 €

# 5 — index=True pour garder la colonne Ville dans le fichier
pivot_qte.to_csv("ventes_pivot.csv", index=True)
print("✅ Exporté")
```
</details>

---
## 🧩 Exercice 2 – Fusion de tableaux

Vous gérez une école avec trois fichiers séparés :

```python
etudiants = pd.DataFrame({
    "id"    : [1, 2, 3, 4, 5],
    "Nom"   : ["Alice", "Bob", "Chloé", "David", "Eva"],
    "Promo" : ["2023", "2023", "2024", "2024", "2023"]
})

notes = pd.DataFrame({
    "id"    : [1, 2, 3, 6],       # David (4) et Eva (5) absents, id 6 inconnu
    "Note"  : [14, 17, 12, 9]
})

stages = pd.DataFrame({
    "id"      : [1, 3, 5],        # seulement Alice, Chloé, Eva ont un stage
    "Entreprise": ["Google", "Apple", "Amazon"]
})
```

1. Fusionnez `etudiants` et `notes` avec **inner** — qui est exclu ?
2. Fusionnez `etudiants` et `notes` avec **left** — qu'arrive-t-il aux étudiants sans note ?
3. Fusionnez `etudiants` et `stages` avec **left** — tous les étudiants doivent apparaître
4. À partir de la jointure left (exo 3), combien d'étudiants **n'ont pas de stage** ? (indice : `isna()`)
5. Exportez la jointure complète (etudiants + notes + stages) en Excel sous `"ecole.xlsx"`

> 💡 **Indice exo 5 :** pour fusionner 3 tableaux, faites deux `merge()` successifs : d'abord étudiants+notes, puis le résultat avec stages.

In [ ]:
import pandas as pd

etudiants = pd.DataFrame({
    "id"   : [1, 2, 3, 4, 5],
    "Nom"  : ["Alice", "Bob", "Chloé", "David", "Eva"],
    "Promo": ["2023", "2023", "2024", "2024", "2023"]
})

notes = pd.DataFrame({
    "id"  : [1, 2, 3, 6],
    "Note": [14, 17, 12, 9]
})

stages = pd.DataFrame({
    "id"        : [1, 3, 5],
    "Entreprise": ["Google", "Apple", "Amazon"]
})

# TODO 1 : inner — etudiants + notes
inner = pd.merge(etudiants, notes, on="id", how=...)
print("1. Inner (etudiants + notes) :")
print(inner)
print(f"→ {5 - len(inner)} étudiant(s) exclu(s) + 1 note sans étudiant exclue")

# TODO 2 : left — etudiants + notes
left_notes = pd.merge(etudiants, notes, on="id", how=...)
print("\n2. Left (tous les étudiants) :")
print(left_notes)

# TODO 3 : left — etudiants + stages
left_stages = pd.merge(etudiants, stages, on="id", how=...)
print("\n3. Left (étudiants + stages) :")
print(left_stages)

# TODO 4 : combien sans stage ?
sans_stage = left_stages["Entreprise"].isna().sum()
print(f"\n4. Étudiants sans stage : {sans_stage}")

# TODO 5 : fusion complète puis export Excel
complet = pd.merge(left_notes, stages, on="id", how=...)
complet.to_excel(..., index=False)
print("\n5. ✅ Exporté en Excel : ecole.xlsx")
print(complet)

<details>
<summary>💡 Correction</summary>

```python
import pandas as pd

etudiants = pd.DataFrame({
    "id"   : [1, 2, 3, 4, 5],
    "Nom"  : ["Alice", "Bob", "Chloé", "David", "Eva"],
    "Promo": ["2023", "2023", "2024", "2024", "2023"]
})
notes  = pd.DataFrame({"id": [1, 2, 3, 6], "Note": [14, 17, 12, 9]})
stages = pd.DataFrame({"id": [1, 3, 5], "Entreprise": ["Google", "Apple", "Amazon"]})

# 1 — inner garde seulement id 1, 2, 3 (communs aux deux)
inner = pd.merge(etudiants, notes, on="id", how="inner")
print(inner)
# David (4) et Eva (5) exclus (pas dans notes)
# id 6 exclu (pas dans etudiants)

# 2 — left garde tous les étudiants
left_notes = pd.merge(etudiants, notes, on="id", how="left")
print(left_notes)
# David et Eva apparaissent avec Note = NaN

# 3
left_stages = pd.merge(etudiants, stages, on="id", how="left")
print(left_stages)
# Bob, David ont Entreprise = NaN

# 4
sans_stage = left_stages["Entreprise"].isna().sum()
print(f"Étudiants sans stage : {sans_stage}")  # 2 (Bob et David)

# 5 — deux merge successifs : etudiants + notes puis + stages
complet = pd.merge(left_notes, stages, on="id", how="left")
complet.to_excel("ecole.xlsx", index=False)
print(complet)
# Résultat final : 5 lignes, tous les étudiants,
# avec leur note (NaN si absente) et leur stage (NaN si absent)
```
</details>

---
## ✅ Récapitulatif

| Concept | Code | Ce qu'il faut retenir |
|---------|------|-----------------------|
| **`pivot_table()`** | `df.pivot_table(values, index, columns, aggfunc)` | TCD Pandas — croise deux dimensions avec un calcul |
| **`fill_value=0`** | paramètre de `pivot_table` | Remplace les cases vides par 0 |
| **`margins=True`** | paramètre de `pivot_table` | Ajoute une ligne/colonne de totaux |
| **`merge()` inner** | `pd.merge(A, B, on="id", how="inner")` | Garde seulement les lignes présentes dans **les deux** |
| **`merge()` left** | `how="left"` | Garde tout A — lignes sans correspondance → NaN |
| **`merge()` outer** | `how="outer"` | Garde tout des deux côtés — manquants → NaN |
| **Export CSV** | `df.to_csv("f.csv", index=False)` | Format universel — `index=False` évite la colonne parasite |
| **Export Excel** | `df.to_excel("f.xlsx", index=False)` | Pour partager avec des collègues Excel |
| **Export JSON** | `df.to_json("f.json", orient="records")` | Pour les APIs web |

**Règle pour choisir le type de jointure :**
- Je veux **seulement les correspondances** → `inner`
- Je veux **tout conserver** à gauche (tableau principal) → `left`
- Je veux **absolument tout** des deux côtés → `outer`

---
📘 **Prochain notebook → `05` : Visualisation avec Pandas**